In [ ]:
# Cell 1: Setup and Library Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting configuration for clean visualizations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

print("All data analytics and visualization libraries imported successfully!")

In [ ]:
# Cell 2: Project 1 - COVID-19 Dataset Generation & Cleaning
np.random.seed(42)

dates = pd.date_range(start="2021-01-01", periods=180, freq="D")
countries = ["India", "USA", "Brazil", "UK", "Germany"]

data_list = []
for country in countries:
    base_cases = np.random.randint(1000, 50000)
    for date in dates:
        confirmed = int(base_cases + np.random.normal(500, 200))
        recovered = int(confirmed * np.random.uniform(0.80, 0.95))
        deaths = int(confirmed * np.random.uniform(0.01, 0.03))
        vaccines = int(np.random.uniform(5000, 100000))
        
        data_list.append({
            "Date": date,
            "Country": country,
            "Confirmed": confirmed,
            "Recovered": recovered,
            "Deaths": deaths,
            "Vaccinators_Daily": vaccines
        })
        base_cases += np.random.randint(-100, 600)

df_covid = pd.DataFrame(data_list)

# Injecting artificial missing values for EDA demonstration
df_covid.loc[df_covid.sample(frac=0.05, random_state=42).index, "Recovered"] = np.nan
df_covid.loc[df_covid.sample(frac=0.03, random_state=42).index, "Deaths"] = np.nan

print("--- Initial Missing Values ---")
print(df_covid.isnull().sum())

# Handling Missing Values (Forward fill / Median imputation per country)
df_covid["Recovered"] = df_covid.groupby("Country")["Recovered"].transform(lambda x: x.fillna(x.median()))
df_covid["Deaths"] = df_covid.groupby("Country")["Deaths"].transform(lambda x: x.fillna(x.median()))

df_covid.to_csv("covid19_analysis_data.csv", index=False)
print("\nMissing values after cleaning:")
print(df_covid.isnull().sum())
print("\nDataset Shape:", df_covid.shape)
df_covid.head()

In [ ]:
# Cell 3: Project 1 - COVID-19 Trend Analysis & Country Statistics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Daily Confirmed Cases Trend by Country
sns.lineplot(data=df_covid, x="Date", y="Confirmed", hue="Country", ax=axes[0, 0], ci=None)
axes[0, 0].set_title("1. Daily Confirmed Cases Trend")
axes[0, 0].set_ylabel("Confirmed Cases")

# 2. Total Cases by Country (Bar Chart)
country_totals = df_covid.groupby("Country")[["Confirmed", "Recovered", "Deaths"]].sum().reset_index()
sns.barplot(data=country_totals, x="Country", y="Confirmed", palette="magma", ax=axes[0, 1])
axes[0, 1].set_title("2. Total Confirmed Cases by Country")
axes[0, 1].set_ylabel("Total Confirmed")

# 3. Death Rate vs Recovery Rate Comparison
country_totals["Mortality_Rate_%"] = (country_totals["Deaths"] / country_totals["Confirmed"]) * 100
sns.barplot(data=country_totals, x="Country", y="Mortality_Rate_%", palette="Reds", ax=axes[1, 0])
axes[1, 0].set_title("3. Mortality Rate (%) by Country")
axes[1, 0].set_ylabel("Mortality Rate (%)")

# 4. Vaccination Trends
sns.lineplot(data=df_covid, x="Date", y="Vaccinators_Daily", hue="Country", ax=axes[1, 1], ci=None)
axes[1, 1].set_title("4. Daily Vaccination Administration Trend")
axes[1, 1].set_ylabel("Daily Vaccines Administered")

plt.tight_layout()
plt.show()

print("--- COVID-19 Country Summary ---")
print(country_totals.to_string(index=False))

In [ ]:
# Cell 4: Project 2 - Sales Data Generation
np.random.seed(101)
n_sales = 1000

dates = pd.date_range(start="2025-01-01", end="2025-12-31", periods=n_sales)
regions = ["North", "South", "East", "West"]
categories = ["Electronics", "Office Supplies", "Furniture"]
segments = ["Consumer", "Corporate", "Home Office"]

products = {
    "Electronics": ["Laptop", "Smartphone", "Headphones"],
    "Office Supplies": ["Binder", "Paper", "Pens"],
    "Furniture": ["Chair", "Desk", "Bookcase"]
}

sales_data = []
for i in range(n_sales):
    cat = np.random.choice(categories)
    prod = np.random.choice(products[cat])
    reg = np.random.choice(regions)
    seg = np.random.choice(segments)
    
    unit_price = np.random.uniform(10, 800)
    quantity = np.random.randint(1, 10)
    sales = unit_price * quantity
    margin = np.random.uniform(0.1, 0.4)
    profit = sales * margin
    
    sales_data.append({
        "Order_Date": np.random.choice(dates),
        "Region": reg,
        "Category": cat,
        "Product": prod,
        "Segment": seg,
        "Sales": round(sales, 2),
        "Profit": round(profit, 2),
        "Quantity": quantity
    })

df_sales = pd.DataFrame(sales_data)
df_sales["Month"] = pd.to_datetime(df_sales["Order_Date"]).dt.to_period("M").astype(str)

df_sales.to_csv("sales_analysis_data.csv", index=False)
print("Project 2 Dataset Shape:", df_sales.shape)
df_sales.head()

In [ ]:
# Cell 5: Project 2 - Sales & Profit Analytics Dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. Monthly Sales & Profit Performance
monthly_perf = df_sales.groupby("Month")[["Sales", "Profit"]].sum().reset_index()
sns.lineplot(data=monthly_perf, x="Month", y="Sales", marker="o", label="Sales", ax=axes[0, 0], color="teal")
sns.lineplot(data=monthly_perf, x="Month", y="Profit", marker="s", label="Profit", ax=axes[0, 0], color="orange")
axes[0, 0].set_title("1. Monthly Sales & Profit Trend (2025)")
axes[0, 0].set_xticklabels(monthly_perf["Month"], rotation=45)
axes[0, 0].set_ylabel("Amount ($)")

# 2. Sales by Customer Segment
sns.barplot(data=df_sales, x="Segment", y="Sales", hue="Category", estimator=sum, ci=None, ax=axes[0, 1], palette="Blues")
axes[0, 1].set_title("2. Total Sales by Customer Segment & Category")
axes[0, 1].set_ylabel("Total Sales ($)")

# 3. Regional Sales & Profit Share
region_perf = df_sales.groupby("Region")[["Sales", "Profit"]].sum().reset_index()
sns.barplot(data=region_perf, x="Region", y="Sales", ax=axes[1, 0], palette="viridis")
axes[1, 0].set_title("3. Total Revenue by Region")
axes[1, 0].set_ylabel("Sales ($)")

# 4. Product Category Profitability Heatmap
pivot_prod = df_sales.pivot_table(index="Category", columns="Region", values="Profit", aggfunc="sum")
sns.heatmap(pivot_prod, annot=True, fmt=".1f", cmap="YlGnBu", ax=axes[1, 1])
axes[1, 1].set_title("4. Profit Distribution Heatmap (Category vs Region)")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Project 2 - Business Insight Summaries
print("=========================================================")
print("             EXECUTIVE BUSINESS INSIGHTS                 ")
print("=========================================================")

top_category = df_sales.groupby("Category")["Sales"].sum().idxmax()
top_region = df_sales.groupby("Region")["Sales"].sum().idxmax()
top_segment = df_sales.groupby("Segment")["Profit"].sum().idxmax()

total_sales = df_sales["Sales"].sum()
total_profit = df_sales["Profit"].sum()
overall_margin = (total_profit / total_sales) * 100

print(f"• Total Annual Revenue Generated: ${total_sales:,.2f}")
print(f"• Total Net Profit: ${total_profit:,.2f} (Profit Margin: {overall_margin:.2f}%)")
print(f"• Top Performing Product Category: {top_category}")
print(f"• Highest Revenue Generating Region: {top_region}")
print(f"• Most Profitable Customer Segment: {top_segment}")
print("=========================================================")